# Análisis exploratorio básico – OT Consulting (versión individual)
Exploración sencilla de los Excel del ERP de OTC: qué datos hay, cuántos faltan y qué se ve a primera vista.

## 1. Importar los archivos

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

archivos = ['Catalogo_Colaboradores.xlsx','Sueldos_Salarios_OTC.xlsx','Asignaciones_Colaboradores.xlsx','Catalogo_Clientes.xlsx',
            'Catalogo_Activos.xlsx','Inversion_Activos_OTC.xlsx','Catalogo_Roles.xlsx','Catalogo_Departamentos.xlsx',
            'Catalogo_Habilidades.xlsx','Matriz_Habilidades.xlsx']

# Si no se han subido, Colab abre el cuadro para subir los archivos
if not all(os.path.exists('/content/' + a) for a in archivos):
    from google.colab import files
    files.upload()

datos = {a.replace('.xlsx',''): pd.read_excel('/content/' + a) for a in archivos}
colab = datos['Catalogo_Colaboradores']; sueldos = datos['Sueldos_Salarios_OTC']; asig = datos['Asignaciones_Colaboradores']
clientes = datos['Catalogo_Clientes']; activos = datos['Catalogo_Activos']; matriz = datos['Matriz_Habilidades']
print('Se cargaron', len(datos), 'archivos')

## 2. Resumen de cada tabla

In [ ]:
resumen = pd.DataFrame({
    'filas': [d.shape[0] for d in datos.values()],
    'columnas': [d.shape[1] for d in datos.values()],
    '% de celdas vacías': [round(d.isna().mean().mean()*100, 1) for d in datos.values()],
    'filas repetidas': [int(d.duplicated().sum()) for d in datos.values()],
}, index=datos.keys())
resumen

In [ ]:
# Primeras filas de las tablas principales
colab.iloc[:, :12].head()

In [ ]:
# Columnas con más datos vacíos en el catálogo de colaboradores
(colab.isna().mean()*100).round(1).sort_values(ascending=False).head(10).to_frame('% vacío')

## 3. Colaboradores

In [ ]:
print(colab['Estatus'].value_counts())
activos_c = colab[colab['Estatus'] == 'Activo']
print('\nPuestos más comunes entre activos:')
activos_c['Puesto'].value_counts().head(8)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
colab['Estatus'].value_counts().plot.bar(ax=ax[0], rot=0, color=['green','red']); ax[0].set_title('Colaboradores por estatus')
activos_c['Tipo Nómina'].fillna('Sin dato').value_counts().plot.barh(ax=ax[1]); ax[1].set_title('Activos por tipo de nómina')
plt.tight_layout(); plt.show()

## 4. Sueldos

In [ ]:
print('Colaboradores activos con sueldo:', len(sueldos))
print('Costo mensual total (Sueldo Total Integrado): $', f"{sueldos['Sueldo Total Integrado'].sum():,.0f}")
sueldos['Sueldo Total Integrado'].describe().round(0).to_frame('MXN')

In [ ]:
# Costo por hora = Sueldo Total Integrado / 160 horas al mes (supuesto)
sueldos['Costo por hora'] = (sueldos['Sueldo Total Integrado'] / 160).round(1)
sueldos[sueldos['Costo por hora'] > 0]['Costo por hora'].plot.hist(bins=12, figsize=(7,4), color='steelblue')
plt.title('Costo por hora de los activos'); plt.xlabel('MXN por hora'); plt.show()

## 5. Asignaciones a clientes

In [ ]:
print('Colaboradores con asignación:', asig['ID Colaborador'].nunique(), 'de', len(sueldos), 'activos')
asig.groupby(['Cliente','Proyecto']).size().to_frame('personas')

In [ ]:
asig.groupby('Cliente').size().sort_values().plot.barh(figsize=(7,3), color='steelblue'); plt.title('Personas asignadas por cliente'); plt.show()

## 6. Activos (equipos)

In [ ]:
print('Equipos:', len(activos), '| Inversión registrada: $', f"{activos['Costo'].sum():,.0f}")
print('Equipos con costo 0:', int((activos['Costo'] == 0).sum()))
print('Equipos sin asignar:', int((activos['Colaborador Asignado'] == 'Sin Asignar').sum()))
activos['Marca'].str.strip().str.upper().value_counts().to_frame('equipos')

## 7. Habilidades

In [ ]:
columnas_hab = [c for c in matriz.columns if ' - ' in c]
niveles = pd.Series(matriz[columnas_hab].values.ravel()).value_counts()
print('Personas en la matriz:', len(matriz), '| Habilidades:', len(columnas_hab))
niveles.reindex(['Básico','Intermedio','Avanzado','Experto']).plot.bar(rot=0, figsize=(6,3.5), color='steelblue'); plt.title('Niveles de habilidad registrados'); plt.show()

## 8. Problemas encontrados en los datos

In [ ]:
act = colab[colab['Estatus'] == 'Activo']
problemas = pd.DataFrame([
 ['Colaboradores','Puesto = PENDIENTE (activos)',      int((act['Puesto']=='PENDIENTE').sum()), len(act)],
 ['Colaboradores','Departamento vacío',                 int(colab['Departamento'].isna().sum()), len(colab)],
 ['Colaboradores','Baja sin fecha de baja',             int(((colab['Estatus']=='Baja') & colab['Fecha de Baja'].isna()).sum()), int((colab['Estatus']=='Baja').sum())],
 ['Colaboradores','Activos con sueldo 0',               int((act['Sueldo Total Integrado']==0).sum()), len(act)],
 ['Asignaciones', 'Rol asignado = PENDIENTE',           int((asig['Rol Asignado']=='PENDIENTE').sum()), len(asig)],
 ['Asignaciones', 'Fecha de inicio 1990-01-01',         int((asig['Fecha Inicio']=='1990-01-01').sum()), len(asig)],
 ['Asignaciones', 'Activos sin asignación',             len(sueldos) - asig['ID Colaborador'].nunique(), len(sueldos)],
 ['Activos',      'Costo = 0',                          int((activos['Costo']==0).sum()), len(activos)],
 ['Activos',      'Marca escrita de varias formas',     activos['Marca'].nunique() - activos['Marca'].str.strip().str.upper().nunique(), activos['Marca'].nunique()],
 ['Clientes',     'Proyectos sin PM de OTC',            int(clientes['PM OTC'].isna().sum()), len(clientes)],
 ['Clientes',     'Proyectos sin fecha fin',            int(clientes['Fecha Fin'].isna().sum()), len(clientes)],
], columns=['tabla','problema','casos','de_un_total'])
problemas['%'] = (problemas['casos'] / problemas['de_un_total'] * 100).round(0)
problemas

## 9. Conclusiones rápidas
- Hay 107 colaboradores, 75 activos y 32 bajas.
- Muchos campos clave (puesto, departamento, rol asignado, fechas) están en "PENDIENTE" o vacíos, así que antes de analizar costos por rol o por proyecto hay que limpiarlos.
- Solo 30 de los 75 activos tienen asignación registrada a un cliente.
- Más de la mitad de los equipos no tiene costo registrado.